# Annotation Pipeline
Runs **Llama 3.3 70B** and **DeepSeek-V3** on all documents in `annotation_dataset.csv`.  
Appends JSON scores as new columns and saves to `open_annotation_dataset.csv`.

**New columns:**  
`llama_Inflation`, `llama_Labor`, `llama_Growth`, `llama_Financial_Stability`, `llama_Policy_Layer`, `llama_Policy_Disagreement`  
`ds_Inflation`, `ds_Labor`, `ds_Growth`, `ds_Financial_Stability`, `ds_Policy_Layer`, `ds_Policy_Disagreement`

In [1]:
import pandas as pd
import json
import re
import time
from openai import OpenAI
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
API_KEY = "YOUR_TOGETHER_AI_KEY"   # ← replace with your key 
INPUT_CSV  = "annotation_dataset.csv"
OUTPUT_CSV = "open_annotation_dataset.csv"

MODELS = {
    "llama": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    "ds":    "deepseek-ai/DeepSeek-V3",
}

# Doc types that run each prompt
POLICY_TYPES   = {"statement", "press_conference"}
DISAGREE_TYPES = {"minutes", "press_conference"}
# All doc types run macro_sentiment

SLEEP_BETWEEN_CALLS = 0.5

client = OpenAI(api_key=API_KEY, base_url="https://api.together.xyz/v1")
token_tracker = defaultdict(lambda: {"input_tokens": 0, "output_tokens": 0, "calls": 0})

print("Config loaded.")

Config loaded.


## Prompts

In [2]:
PROMPTS = {

"macro_sentiment": """You are a monetary policy text analysis assistant. Your task is to extract structured macroeconomic signals from an FOMC communication document by evaluating economic condition language only. You must strictly avoid any interpretation related to monetary policy stance, policy tools, interest rate decisions, or forward guidance. The objective is to isolate and quantify information that reflects the state of the economy itself, rather than the policy response. All outputs must be grounded in the text. Do not infer information that is not explicitly supported.

The analysis follows a fixed sequence that must be applied consistently across all topics. For each topic, you must first determine whether it is mentioned in the document. If it is mentioned, you must then assess its importance and evaluate two dimensions: the current condition and the outlook. If a topic is not mentioned, all subsequent fields must be set to null.

The evaluation must be performed for four topics: Inflation, Labor Market, Growth, and Financial Stability. Each topic must be analyzed independently, and signals from one topic must not influence another.

Global Interpretation Rules:
1. Dominant narrative rule: When multiple or conflicting signals appear, identify the overall qualitative direction of the text and assign a score based on that dominant narrative. Do not average positive and negative signals.
2. No over-inference: Only use information explicitly stated in the document. If a signal is not clearly supported, it must be treated as missing.
3. Strict separation of dimensions: Current condition reflects the present state; outlook reflects forward-looking expectations. Do not mix them.
4. Null discipline: If a dimension is not described, return null instead of guessing.

Topic Definitions and Sign Conventions:
- Inflation: Above target or increasing -> positive; Below target or decreasing -> negative; Near target or without a clear directional trend -> neutral. If level and direction conflict, follow the dominant narrative.
- Labor Market: Strong or tight -> positive; Weak or slack -> negative.
- Growth: Strong or expanding -> positive; Weak or slowing -> negative.
- Financial Stability: Stable or resilient -> positive; Fragile or stressed -> negative. Tight financial conditions -> negative; Loose financial conditions -> positive. Explicitly excludes monetary policy stance, interest rate levels in isolation, housing conditions, and forward guidance.

Step-by-Step Evaluation:
Step 1 - Topic Mention: 1 = mentioned, 0 = not mentioned. If 0, set all remaining fields to null.
Step 2 - Topic Importance: "Major" = central theme, discussed in detail or repeatedly; "Minor" = briefly mentioned or supporting context.
Step 3A - Current Condition: Reflects the overall strength and intensity of the current state, conditional on the direction (positive, negative, or neutral) already determined by the topic-specific definitions above.
+2 = very strong positive condition, +1 = moderately positive condition, 0 = neutral/balanced/no clear directional signal, -1 = moderately negative condition, -2 = very weak negative condition. Return null if no clear description of the current state is provided.
Step 3B - Outlook: Reflects the direction and strength of expected change, conditional on the direction (positive, negative, or neutral) defined by the topic-specific rules above.
+2 = strongly improving, +1 = moderately improving, 0 = neutral/balanced/no clear directional signal, -1 = moderately worsening, -2 = strongly worsening. Return null if no forward-looking content is explicitly present.

Return results in valid JSON format. Use null for missing values. Do not include any additional commentary outside the JSON output.

{
  "Inflation": {
    "Mention": value,
    "Importance": "Major" or "Minor" or null,
    "Current_Condition": value or null,
    "Outlook": value or null
  },
  "Labor": {
    "Mention": value,
    "Importance": "Major" or "Minor" or null,
    "Current_Condition": value or null,
    "Outlook": value or null
  },
  "Growth": {
    "Mention": value,
    "Importance": "Major" or "Minor" or null,
    "Current_Condition": value or null,
    "Outlook": value or null
  },
  "Financial_Stability": {
    "Mention": value,
    "Importance": "Major" or "Minor" or null,
    "Current_Condition": value or null,
    "Outlook": value or null
  },
  "Short_Justification": "Brief explanation (maximum 4-5 sentences)"
}

Document:
""",

"policy_layer": """You are a monetary policy text analysis assistant. Your task is to extract policy-layer signals from an FOMC communication document. The goal is to quantify how the Federal Reserve communicates its current policy stance and the expected direction of future policy.

You must evaluate policy language only. Do not infer policy signals that are not explicitly supported by the text.

Scope of Valid Signals: Only language that directly reflects policy or policy expectations. This includes policy decisions, descriptions of policy tools, the reaction function, the future path of policy, and explicit statements about risk asymmetry between employment and inflation. Macroeconomic condition descriptions must be excluded unless they are explicitly framed as asymmetric risks between the dual mandate goals (employment vs. inflation) — in such cases, the asymmetric risk framing should be treated as a policy signal contributing to the Hawkish-Dovish stance. Language that describes the economy without policy implications must not be used. If the document does not contain relevant language for a given category, assign null.

Hawkish-Dovish Stance (Current Policy Position):
Captures whether policy is restrictive or accommodative.
Scale: +2 = strongly hawkish (clear tightening bias or strong emphasis on restrictiveness), +1 = moderately hawkish, 0 = neutral or balanced, -1 = moderately dovish, -2 = strongly dovish (clear easing bias or strong emphasis on accommodation). If no policy stance language is present: null.

Forward Guidance (Directional Policy Signal):
Captures the direction of expected change in policy. Must rely only on forward-looking policy language. Forward Guidance reflects the direction of movement from the current stance, not the eventual level of policy. Statements that are purely data-dependent without a directional implication must be treated as neutral.
Scale: +2 = strong signal of further tightening, +1 = moderate signal of tightening, 0 = no directional signal or purely data-dependent, -1 = moderate signal of easing, -2 = strong signal of further easing. If no forward-looking policy language is present: null.

Return valid JSON. Use null for missing values. No additional commentary outside the JSON structure.

{
  "Hawkish_Dovish_Stance": value or null,
  "Forward_Guidance": value or null,
  "Short_Justification": "Brief explanation (maximum 2 sentences)"
}

Document:
""",

"policy_disagreement": """You are a monetary policy communication analyst. Your task is to extract the degree of internal policy disagreement from an FOMC communication document. This analysis applies only to FOMC Minutes and FOMC Press Conferences.

Definition: Internal Policy Disagreement captures whether participants hold different views on the direction or implementation of monetary policy. Focus on policy-relevant divergence, including: explicit differences in policy path (direction, pace, timing, magnitude), and implicit differences in risk assessment or economic interpretation that imply different policy preferences. Linguistic signals such as "some participants," "others," or "participants differed" may indicate disagreement but are not sufficient on their own — you must determine whether the divergence implies different policy choices.

Exclusion Rules (Strict) - do NOT treat the following as disagreement:
- General uncertainty
- Routine data-dependence language
- Differences in economic descriptions without policy implications
If divergence does not imply different policy paths, assign 0.

Scoring Framework:
0 = No Disagreement: Language is unified, with no indication of differing policy-relevant views.
1 = Implicit Disagreement: Participants differ in risk assessments or economic interpretations in ways that imply different policy leanings, but no explicit disagreement on policy actions is stated.
2 = Explicit Policy Disagreement: Participants explicitly express different views on the policy path (direction, pace, timing, or magnitude of actions).

Decision Rule (Strict Hierarchy): If any explicit policy disagreement appears anywhere -> assign 2. Else if only implicit disagreement -> assign 1. Else -> assign 0. Do not average signals. The highest level observed determines the final score.

Return valid JSON. No additional commentary outside the JSON structure.

{
  "Policy_Disagreement": value,
  "Short_Justification": "Brief explanation (maximum 2 sentences)"
}

Document:
""",
}

print("Prompts loaded.")

Prompts loaded.


## Core Functions

In [3]:
def call_model(model_key, model_id, prompt_key, doc_text):
    max_tokens = 8000 if "qwen" in model_key else 1000
    full_prompt = PROMPTS[prompt_key] + doc_text
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": full_prompt}],
            max_tokens=max_tokens,
            temperature=0,
        )
        usage = response.usage
        token_tracker[model_key]["input_tokens"]  += usage.prompt_tokens
        token_tracker[model_key]["output_tokens"] += usage.completion_tokens
        token_tracker[model_key]["calls"]         += 1

        raw   = response.choices[0].message.content.strip()
        clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
        clean = clean.replace("```json", "").replace("```", "").strip()
        clean = re.sub(r':\s*\+(\d)', r': \1', clean)
        clean = re.sub(r':\s*NA\b', ': null', clean)
        parsed = json.loads(clean)
        return {"success": True, "parsed": parsed,
                "input_tokens": usage.prompt_tokens,
                "output_tokens": usage.completion_tokens}
    except json.JSONDecodeError as e:
        print(f"      JSON error: {e}")
        return {"success": False, "error": str(e), "parsed": None}
    except Exception as e:
        print(f"      API error: {e}")
        return {"success": False, "error": str(e), "parsed": None}


def annotate_doc(doc_type, doc_text, model_key, model_id):
    out = {
        "Inflation": None, "Labor": None,
        "Growth": None, "Financial_Stability": None,
        "Policy_Layer": None, "Policy_Disagreement": None,
    }
    # Macro sentiment — all doc types
    r = call_model(model_key, model_id, "macro_sentiment", doc_text)
    time.sleep(SLEEP_BETWEEN_CALLS)
    if r["success"] and r["parsed"]:
        for topic in ["Inflation", "Labor", "Growth", "Financial_Stability"]:
            if topic in r["parsed"]:
                out[topic] = json.dumps(r["parsed"][topic])

    # Policy layer — statement and press_conference only
    if doc_type in POLICY_TYPES:
        r = call_model(model_key, model_id, "policy_layer", doc_text)
        time.sleep(SLEEP_BETWEEN_CALLS)
        if r["success"] and r["parsed"]:
            out["Policy_Layer"] = json.dumps(r["parsed"])

    # Policy disagreement — minutes and press_conference only
    if doc_type in DISAGREE_TYPES:
        r = call_model(model_key, model_id, "policy_disagreement", doc_text)
        time.sleep(SLEEP_BETWEEN_CALLS)
        if r["success"] and r["parsed"]:
            out["Policy_Disagreement"] = json.dumps(r["parsed"])

    return out

print("Functions ready.")

Functions ready.


---
## Section 1 · Load Dataset

In [4]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} documents")
print()
print(df['doc_type'].value_counts().to_string())

# Initialise new columns
new_cols = [
    "llama_Inflation", "llama_Labor", "llama_Growth", "llama_Financial_Stability",
    "llama_Policy_Layer", "llama_Policy_Disagreement",
    "ds_Inflation", "ds_Labor", "ds_Growth", "ds_Financial_Stability",
    "ds_Policy_Layer", "ds_Policy_Disagreement",
]
for col in new_cols:
    if col not in df.columns:
        df[col] = None

print(f"\nNew columns initialised: {new_cols}")

Loaded 61 documents

doc_type
statement               20
minutes                 10
mpr                      5
speech                   5
press_conference         5
beigebook                5
implementation_notes     5
testimony                5
press_release            1

New columns initialised: ['llama_Inflation', 'llama_Labor', 'llama_Growth', 'llama_Financial_Stability', 'llama_Policy_Layer', 'llama_Policy_Disagreement', 'ds_Inflation', 'ds_Labor', 'ds_Growth', 'ds_Financial_Stability', 'ds_Policy_Layer', 'ds_Policy_Disagreement']


---
## Section 2 · Run Annotation Pipeline
Saves a checkpoint to `open_annotation_dataset.csv` after every document.  
If interrupted, re-run this cell — already-annotated rows will be skipped.

In [5]:
total = len(df)

for idx, row in df.iterrows():
    doc_id   = row["doc_id"]
    doc_type = row["doc_type"]
    doc_text = str(row["text"])

    # Skip if already annotated (checkpoint resume)
    if pd.notna(df.at[idx, "llama_Inflation"]):
        print(f"[{idx+1}/{total}] doc_id={doc_id} — already done, skipping")
        continue

    print(f"\n[{idx+1}/{total}] doc_id={doc_id} | {doc_type} | {len(doc_text):,} chars")

    for model_key, model_id in MODELS.items():
        prefix = "llama" if model_key == "llama" else "ds"
        print(f"  {model_key}...", end=" ", flush=True)

        scores = annotate_doc(doc_type, doc_text, model_key, model_id)

        df.at[idx, f"{prefix}_Inflation"]           = scores["Inflation"]
        df.at[idx, f"{prefix}_Labor"]               = scores["Labor"]
        df.at[idx, f"{prefix}_Growth"]              = scores["Growth"]
        df.at[idx, f"{prefix}_Financial_Stability"] = scores["Financial_Stability"]
        df.at[idx, f"{prefix}_Policy_Layer"]        = scores["Policy_Layer"]
        df.at[idx, f"{prefix}_Policy_Disagreement"] = scores["Policy_Disagreement"]

        print("done")

    # Checkpoint save after every document
    df.to_csv(OUTPUT_CSV, index=False)

print("\n✓ All documents annotated.")


[1/61] doc_id=1071 | statement | 2,736 chars
doneama... 
done... 

[2/61] doc_id=1514 | statement | 2,077 chars
doneama... 
done... 

[3/61] doc_id=91 | statement | 1,637 chars
doneama... 
done... 

[4/61] doc_id=1198 | statement | 4,117 chars
doneama... 
done... 

[5/61] doc_id=1093 | statement | 1,706 chars
doneama... 
done... 

[6/61] doc_id=870 | statement | 5,106 chars
doneama... 
done... 

[7/61] doc_id=662 | statement | 11,284 chars
doneama... 
done... 

[8/61] doc_id=149 | statement | 1,868 chars
doneama... 
done... 

[9/61] doc_id=386 | statement | 9,435 chars
doneama... 
done... 

[10/61] doc_id=433 | statement | 10,250 chars
doneama... 
done... 

[11/61] doc_id=874 | statement | 5,119 chars
doneama... 
done... 

[12/61] doc_id=1129 | statement | 1,883 chars
doneama... 
done... 

[13/61] doc_id=983 | statement | 3,571 chars
doneama... 
done... 

[14/61] doc_id=800 | statement | 4,016 chars
doneama... 
done... 

[15/61] doc_id=277 | statement | 2,082 chars
doneama... 
done...

---
## Section 3 · Token Usage & Cost

In [6]:
PRICE_PER_1M = {
    "llama": {"input": 0.18, "output": 0.18},
    "ds":    {"input": 1.25, "output": 1.25},
}

print(f"{'Model':<10} {'Calls':>6} {'Input':>10} {'Output':>10} {'Total':>10} {'Cost':>10}")
print("-" * 58)
for model_key, stats in token_tracker.items():
    total_tok = stats["input_tokens"] + stats["output_tokens"]
    p    = PRICE_PER_1M.get(model_key, {"input": 0, "output": 0})
    cost = (stats["input_tokens"] * p["input"] +
            stats["output_tokens"] * p["output"]) / 1_000_000
    print(f"{model_key:<10} {stats['calls']:>6} "
          f"{stats['input_tokens']:>10,} {stats['output_tokens']:>10,} "
          f"{total_tok:>10,} ${cost:>9.4f}")

Model       Calls      Input     Output      Total       Cost
----------------------------------------------------------
llama         101    543,574     17,717    561,291 $   0.1010
ds            101    541,423     18,587    560,010 $   0.7000


---
## Section 4 · Preview Output

In [7]:
result_df = pd.read_csv(OUTPUT_CSV)
print(f"Output shape: {result_df.shape}")
print()

# Show first annotated row
sample = result_df[result_df['llama_Inflation'].notna()].iloc[0]
print(f"doc_id={sample['doc_id']} | doc_type={sample['doc_type']} | date={sample['doc_date']}")
print()
for col in ["llama_Inflation", "llama_Policy_Layer", "llama_Policy_Disagreement",
            "ds_Inflation",    "ds_Policy_Layer",    "ds_Policy_Disagreement"]:
    val = sample[col]
    if pd.notna(val):
        print(f"  {col}:")
        print(f"    {val}")
    else:
        print(f"  {col}: N/A")
    print()

Output shape: (61, 27)

doc_id=1071 | doc_type=statement | date=2018-03-21

  llama_Inflation:
    {"Mention": 1, "Importance": "Major", "Current_Condition": -1, "Outlook": 1}

  llama_Policy_Layer:
    {"Hawkish_Dovish_Stance": -1, "Forward_Guidance": 1, "Short_Justification": "The Committee's decision to raise the target range for the federal funds rate while stating that the stance of monetary policy remains accommodative suggests a moderately dovish current stance. The expectation of further gradual increases in the federal funds rate provides a moderate signal of tightening."}

  llama_Policy_Disagreement: N/A

  ds_Inflation:
    {"Mention": 1, "Importance": "Major", "Current_Condition": -1, "Outlook": 1}

  ds_Policy_Layer:
    {"Hawkish_Dovish_Stance": -1, "Forward_Guidance": 1, "Short_Justification": "The stance is moderately dovish as the document explicitly states 'The stance of monetary policy remains accommodative'. Forward guidance is moderately hawkish as it signals 'fur

In [13]:
# import numpy as np

df = pd.read_csv(OUTPUT_CSV)

# ── Helper: extract numeric fields from a JSON column ─────────────────────────
def extract_fields(json_str, fields):
    if pd.isna(json_str):
        return {f: np.nan for f in fields}
    try:
        d = json.loads(json_str)
        return {f: d.get(f, np.nan) for f in fields}
    except:
        return {f: np.nan for f in fields}

# ── Define which fields to compare per column type ───────────────────────────
MACRO_FIELDS    = ["Mention", "Importance", "Current_Condition", "Outlook"]
POLICY_FIELDS   = ["Hawkish_Dovish_Stance", "Forward_Guidance"]
DISAGREE_FIELDS = ["Policy_Disagreement"]
TOPIC_COLS      = ["Inflation", "Labor", "Growth", "Financial_Stability"]

# ── Build a flat comparison dataframe ────────────────────────────────────────
records = []

for _, row in df.iterrows():
    doc_id   = row["doc_id"]
    doc_type = row["doc_type"]
    doc_date = row["doc_date"]

    # Macro topics
    for topic in TOPIC_COLS:
        llama_d = extract_fields(row.get(f"llama_{topic}"), MACRO_FIELDS)
        ds_d    = extract_fields(row.get(f"ds_{topic}"),    MACRO_FIELDS)

        for field in ["Current_Condition", "Outlook", "Mention", "Importance"]:
            l_val = llama_d[field]
            d_val = ds_d[field]
            if pd.isna(l_val) and pd.isna(d_val):
                continue
            records.append({
                "doc_id":   doc_id,
                "doc_type": doc_type,
                "doc_date": doc_date,
                "category": topic,
                "field":    field,
                "llama":    l_val,
                "deepseek": d_val,
                "diverge":  str(l_val) != str(d_val),
            })

    # Policy Layer
    llama_pl = extract_fields(row.get("llama_Policy_Layer"), POLICY_FIELDS)
    ds_pl    = extract_fields(row.get("ds_Policy_Layer"),    POLICY_FIELDS)
    for field in POLICY_FIELDS:
        l_val = llama_pl[field]
        d_val = ds_pl[field]
        if pd.isna(l_val) and pd.isna(d_val):
            continue
        records.append({
            "doc_id":   doc_id,
            "doc_type": doc_type,
            "doc_date": doc_date,
            "category": "Policy_Layer",
            "field":    field,
            "llama":    l_val,
            "deepseek": d_val,
            "diverge":  str(l_val) != str(d_val),
        })

    # Policy Disagreement
    llama_pd = extract_fields(row.get("llama_Policy_Disagreement"), DISAGREE_FIELDS)
    ds_pd    = extract_fields(row.get("ds_Policy_Disagreement"),    DISAGREE_FIELDS)
    l_val = llama_pd["Policy_Disagreement"]
    d_val = ds_pd["Policy_Disagreement"]
    if not (pd.isna(l_val) and pd.isna(d_val)):
        records.append({
            "doc_id":   doc_id,
            "doc_type": doc_type,
            "doc_date": doc_date,
            "category": "Policy_Disagreement",
            "field":    "Policy_Disagreement",
            "llama":    l_val,
            "deepseek": d_val,
            "diverge":  str(l_val) != str(d_val),
        })

cmp = pd.DataFrame(records)

# ── Summary 1: Divergence rate by category & field ───────────────────────────
print("=" * 60)
print("DIVERGENCE RATE BY CATEGORY & FIELD")
print("=" * 60)
summary = (
    cmp.groupby(["category", "field"])["diverge"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_diverge", "count": "n_total"})
)
summary["diverge_rate"] = (summary["n_diverge"] / summary["n_total"] * 100).round(1)
summary = summary.sort_values("diverge_rate", ascending=False)
print(summary.to_string())

# ── Summary 2: Divergence rate by doc_type ────────────────────────────────────
print("\n" + "=" * 60)
print("DIVERGENCE RATE BY DOC_TYPE")
print("=" * 60)
by_type = (
    cmp.groupby("doc_type")["diverge"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_diverge", "count": "n_total"})
)
by_type["diverge_rate"] = (by_type["n_diverge"] / by_type["n_total"] * 100).round(1)
by_type = by_type.sort_values("diverge_rate", ascending=False)
print(by_type.to_string())

# ── Summary 3: All divergent cases (numeric fields only) ─────────────────────
print("\n" + "=" * 60)
print("ALL DIVERGENT CASES (numeric fields only)")
print("=" * 60)
numeric_fields = ["Current_Condition", "Outlook", "Hawkish_Dovish_Stance",
                  "Forward_Guidance", "Policy_Disagreement", "Mention"]
divergent = cmp[
    cmp["diverge"] &
    cmp["field"].isin(numeric_fields)
].sort_values(["category", "field", "doc_date"])

print(divergent[["doc_id","doc_type","doc_date","category","field","llama","deepseek"]]
      .to_string(index=False))

print(f"\nTotal divergent cases: {divergent['diverge'].sum()} / {len(cmp)} comparisons")
print(f"Overall divergence rate: {cmp['diverge'].mean()*100:.1f}%")

DIVERGENCE RATE BY CATEGORY & FIELD
                                           n_diverge  n_total  diverge_rate
category            field                                                  
Financial_Stability Outlook                       15       19          78.9
                    Current_Condition             25       40          62.5
                    Importance                    23       42          54.8
Policy_Layer        Forward_Guidance              12       25          48.0
Growth              Current_Condition             22       54          40.7
                    Outlook                       18       49          36.7
Inflation           Current_Condition             18       51          35.3
Labor               Outlook                       14       42          33.3
Policy_Layer        Hawkish_Dovish_Stance          7       25          28.0
Financial_Stability Mention                       16       61          26.2
Inflation           Outlook                       12

### Scores Result Check


In [15]:
row = df[df['doc_id'] == 1309].iloc[0]
for col in ["llama_Inflation", "llama_Labor", "llama_Growth", "llama_Financial_Stability",
            "llama_Policy_Layer", "llama_Policy_Disagreement",
            "ds_Inflation", "ds_Labor", "ds_Growth", "ds_Financial_Stability",
            "ds_Policy_Layer", "ds_Policy_Disagreement"]:
    print(f"{col}:")
    print(f"  {row[col]}")
    print()

llama_Inflation:
  {"Mention": 1, "Importance": "Major", "Current_Condition": 2, "Outlook": -1}

llama_Labor:
  {"Mention": 1, "Importance": "Major", "Current_Condition": 1, "Outlook": 1}

llama_Growth:
  {"Mention": 1, "Importance": "Major", "Current_Condition": 1, "Outlook": 1}

llama_Financial_Stability:
  {"Mention": 1, "Importance": "Minor", "Current_Condition": 1, "Outlook": null}

llama_Policy_Layer:
  nan

llama_Policy_Disagreement:
  nan

ds_Inflation:
  {"Mention": 1, "Importance": "Major", "Current_Condition": -2, "Outlook": -1}

ds_Labor:
  {"Mention": 1, "Importance": "Major", "Current_Condition": 1, "Outlook": 1}

ds_Growth:
  {"Mention": 1, "Importance": "Major", "Current_Condition": 1, "Outlook": 0}

ds_Financial_Stability:
  {"Mention": 0, "Importance": null, "Current_Condition": null, "Outlook": null}

ds_Policy_Layer:
  nan

ds_Policy_Disagreement:
  nan



In [16]:
import pandas as pd
import json
import re
import time
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────────
API_KEY    = "tgp_v1_jBChS7MNIHOS5u98RFNDeRISPqXkqYdJVUmmrJp3Bw8"
INPUT_CSV  = "open_annotation_dataset.csv"   # 已有结果的CSV
OUTPUT_CSV = "open_annotation_dataset.csv"   # 直接更新原文件

MODELS = {
    "llama": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    "ds":    "deepseek-ai/DeepSeek-V3",
}

SLEEP_BETWEEN_CALLS = 0.5

client = OpenAI(api_key=API_KEY, base_url="https://api.together.xyz/v1")

MACRO_PROMPT = PROMPTS["macro_sentiment"]  # 粘贴你的完整macro_sentiment prompt，和pipeline里一样

# ── Helper ────────────────────────────────────────────────────────────────────
def call_macro(model_id, doc_text):
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": MACRO_PROMPT + doc_text}],
            max_tokens=1000,
            temperature=0,
        )
        raw   = response.choices[0].message.content.strip()
        clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
        clean = clean.replace("```json", "").replace("```", "").strip()
        clean = re.sub(r':\s*\+(\d)', r': \1', clean)
        clean = re.sub(r':\s*NA\b', ': null', clean)
        parsed = json.loads(clean)
        return parsed.get("Short_Justification", None)
    except Exception as e:
        print(f"    ERROR: {e}")
        return None

# ── Main ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)

# 初始化新列（如果还没有）
for col in ["llama_Macro_Justification", "ds_Macro_Justification"]:
    if col not in df.columns:
        df[col] = None

total = len(df)

for idx, row in df.iterrows():
    doc_id   = row["doc_id"]
    doc_text = str(row["text"])

    # Skip if already done
    if pd.notna(df.at[idx, "llama_Macro_Justification"]):
        print(f"[{idx+1}/{total}] doc_id={doc_id} — skipping")
        continue

    print(f"[{idx+1}/{total}] doc_id={doc_id}...", end=" ", flush=True)

    for model_key, model_id in MODELS.items():
        prefix = "llama" if model_key == "llama" else "ds"
        justification = call_macro(model_id, doc_text)
        df.at[idx, f"{prefix}_Macro_Justification"] = justification
        time.sleep(SLEEP_BETWEEN_CALLS)

    print("done")

    # Checkpoint
    df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved to {OUTPUT_CSV}")

[1/61] doc_id=1071... done
done1] doc_id=1514... 
done1] doc_id=91... 
done1] doc_id=1198... 
done1] doc_id=1093... 
done1] doc_id=870... 
done1] doc_id=662... 
done1] doc_id=149... 
done1] doc_id=386... 
done61] doc_id=433... 
done61] doc_id=874... 
done61] doc_id=1129... 
done61] doc_id=983... 
done61] doc_id=800... 
done61] doc_id=277... 
done61] doc_id=58... 
done61] doc_id=696... 
done61] doc_id=657... 
done61] doc_id=105... 
done61] doc_id=1223... 
done61] doc_id=113... 
done61] doc_id=75... 
done61] doc_id=769... 
done61] doc_id=990... 
done61] doc_id=497... 
done61] doc_id=728... 
done61] doc_id=1199... 
done61] doc_id=1225... 
done61] doc_id=1090... 
done61] doc_id=1389... 
done61] doc_id=237... 
done61] doc_id=558... 
done61] doc_id=1258... 
done61] doc_id=406... 
done61] doc_id=734... 
done61] doc_id=1260... 
done61] doc_id=22... 
done61] doc_id=1128... 
done61] doc_id=442... 
done61] doc_id=331... 
done61] doc_id=1485... 
done61] doc_id=684... 
done61] doc_id=1379... 
done6

In [17]:
df = pd.read_csv(OUTPUT_CSV)

for doc_id in [1379, 1309, 1514, 728]:
    row = df[df['doc_id'] == doc_id].iloc[0]
    print(f"\n{'='*60}")
    print(f"doc_id={doc_id} | {row['doc_type']} | {row['doc_date']}")
    print(f"{'='*60}")
    print(f"  [llama]  {row['llama_Macro_Justification']}")
    print(f"  [ds]     {row['ds_Macro_Justification']}")


doc_id=1379 | press_conference | 2023-02-01
  [llama]  The document discusses the current state of the economy, with high inflation, a tight labor market, and slow growth. The Fed is committed to bringing inflation back down to 2% and is taking actions to achieve this goal. The labor market remains strong, but growth is subdued due to tighter financial conditions and higher interest rates.
  [ds]     Inflation remains well above target at 5.0% PCE with some recent moderation but still too high. Labor market is extremely tight with robust job gains and low unemployment. Growth slowed significantly with below-trend GDP growth and subdued spending. Financial conditions are mentioned as having tightened significantly but are currently stable.

doc_id=1309 | testimony | 2021-11-30
  [llama]  The document discusses the strengthening economy, improving labor market conditions, and inflation running above the 2 percent goal. It mentions the economy's growth, labor market improvement, and the 

In [18]:
df = pd.read_csv(OUTPUT_CSV)

for doc_id in [1379, 1309, 1514, 728]:
    row = df[df['doc_id'] == doc_id].iloc[0]
    print(f"\n{'='*60}")
    print(f"doc_id={doc_id} | {row['doc_type']} | {row['doc_date']}")
    print(f"{'='*60}")
    
    for col in ["llama_Inflation", "llama_Labor", "llama_Growth", 
                "llama_Financial_Stability", "llama_Policy_Layer", 
                "llama_Policy_Disagreement", "llama_Macro_Justification",
                "ds_Inflation", "ds_Labor", "ds_Growth",
                "ds_Financial_Stability", "ds_Policy_Layer",
                "ds_Policy_Disagreement", "ds_Macro_Justification"]:
        val = row[col]
        if pd.notna(val):
            print(f"\n  {col}:")
            print(f"    {val}")


doc_id=1379 | press_conference | 2023-02-01

  llama_Inflation:
    {"Mention": 1, "Importance": "Major", "Current_Condition": -1, "Outlook": -1}

  llama_Labor:
    {"Mention": 1, "Importance": "Major", "Current_Condition": 2, "Outlook": 0}

  llama_Growth:
    {"Mention": 1, "Importance": "Major", "Current_Condition": -1, "Outlook": 0}

  llama_Financial_Stability:
    {"Mention": 1, "Importance": "Minor", "Current_Condition": -1, "Outlook": null}

  llama_Policy_Layer:
    {"Hawkish_Dovish_Stance": -1, "Forward_Guidance": 1, "Short_Justification": "The FOMC raised the policy interest rate by 25 basis points and continues to anticipate ongoing increases, indicating a moderately dovish current stance with a moderate signal of further tightening."}

  llama_Policy_Disagreement:
    {"Policy_Disagreement": 0, "Short_Justification": "The document does not indicate any explicit or implicit disagreement among participants regarding the policy path, with Chair Powell's statements reflectin